# Implementation Details of the "War" Problem

This lecture focuses on the practical implementation of the "War" problem solution using a modified **Disjoint Set Union (DSU)** data structure. The solution tracks friendship and rivalry relationships through a centralized class.

## 1. Data Structure Components
The core data structure is a modified Union-Find class. It maintains friendship clusters while adding a specific mechanism for enemies.

* **Friends Array:** This is a renamed version of the standard **parent pointer** array. It tracks which people are in the same friendship cluster.
* **Rank Array:** Used for standard **union by rank** optimization to keep the trees balanced.
* **Enemies Array:** A specific array of size $N$ that stores the enemy for each **leader** (root) of a friendship cluster.
    * **Initialization:** Every index in the `enemies` array is set to **-1**, indicating that the person (or cluster) has no known enemies initially.
    * **Mapping:** The indices used are $0$ to $N-1$. 


## 2. Helper Functions (Queries)
The lecture defines two simple methods to answer questions about relationships based on the current state of the data structure.

### areFriends(x, y)
* **Definition:** Checks if two people belong to the same country.
* **Logic:** Determines if the root (leader) of $x$ is the same as the root of $y$.

### areEnemies(x, y)
* **Definition:** Checks if two people belong to different countries.
* **Logic:** Retrieves the root of $x$ and the root of $y$, then checks if the `enemies` entry for one root points to the other root. 

## 3. The `setFriends` Operation
This operation is used to establish that two people are in the same country. It includes several logical branches:

### Contradiction Check
Before any merging, the system checks if `areEnemies(x, y)` is true. If so, it returns **-1**.

### Case Analysis for Merging
1.  **Neither has enemies:** Perform a standard `union(x, y)`.
2.  **One has an enemy:**
    * If $x$ has an enemy $E_x$, after merging $x$ and $y$, $E_x$ becomes the enemy of the new combined cluster.
    * The `enemies` array is updated to ensure the new root of the cluster points to $E_x$.
3.  **Both have enemies:**
    * If $x$ has enemy $E_x$ and $y$ has enemy $E_y$, then $E_x$ and $E_y$ must be friends (Rule: "Common enemy makes friends").
    * Action: Perform `union(x, y)` AND `union(E_x, E_y)`.
    * Action: Update the `enemies` pointer of the new $x+y$ root to point to the new $E_x+E_y$ root. 

## 4. The `setEnemies` Operation
This operation establishes that two people are in different countries.

### Contradiction Check
If `areFriends(x, y)` is already true, the operation returns **-1**.

### Core Logic for `setEnemies`
The most complex case occurs when both people already have existing enemies:
* Let $A$ be the enemy of $x$, and $B$ be the enemy of $y$.
* Establishing $x$ and $y$ as enemies implies that $x$ and $B$ are now friends, and $y$ and $A$ are now friends (Rule: "Enemy of my enemy is my friend").
    1.  Merge sets of $A$ and $y$.
    2.  Merge sets of $B$ and $x$.
    3.  Identify the new roots of these merged sets.
    4.  Update the `enemies` array so the two new roots point to each other.

## 5. Implementation Pseudocode
The following pseudocode outlines the handling of the most complex cases for the two main operations.

```python
Function SetFriends(Person_X, Person_Y):
    Root_X = Find(Person_X)
    Root_Y = Find(Person_Y)

    If areEnemies(Root_X, Root_Y): Return -1

    Enemy_X = EnemyArray[Root_X]
    Enemy_Y = EnemyArray[Root_Y]

    NewRoot_XY = Union(Root_X, Root_Y)

    If Enemy_X != -1 and Enemy_Y != -1:
        NewRoot_Enemy = Union(Enemy_X, Enemy_Y)
        EnemyArray[NewRoot_XY] = NewRoot_Enemy
        EnemyArray[NewRoot_Enemy] = NewRoot_XY
    Else If Enemy_X != -1:
        EnemyArray[NewRoot_XY] = Enemy_X
    Else If Enemy_Y != -1:
        EnemyArray[NewRoot_XY] = Enemy_Y

Function SetEnemies(Person_X, Person_Y):
    Root_X = Find(Person_X)
    Root_Y = Find(Person_Y)

    If areFriends(Root_X, Root_Y): Return -1

    Enemy_X = EnemyArray[Root_X]
    Enemy_Y = EnemyArray[Root_Y]

    If Enemy_X != -1: Union(Enemy_X, Root_Y)
    If Enemy_Y != -1: Union(Enemy_Y, Root_X)
    
    NewRoot_X = Find(Root_X)
    NewRoot_Y = Find(Root_Y)
    EnemyArray[NewRoot_X] = NewRoot_Y
    EnemyArray[NewRoot_Y] = NewRoot_X
```

## Summary of Main Takeaways
* **Modified DSU:** Standard Union-Find can be extended with an auxiliary array to track non-transitive relationships like enmity. 
* **Leader Consistency:** The "Enemy" pointer must always point from the current root of one cluster to the current root of the rival cluster.
* **Recursive Implications:** Setting one relationship (like enmity) can logically force the merging of other sets (friendship) to satisfy the problem's rules. 